In [ ]:
# Given a new microbe, predict what it might consume or secrete based on its taxonomy

# Given a media composition and a list of microbes, predict which microbes will eat which compounds

# move the model and parquet files to repo instead of scratch so they can be commited

# replace pkl with json and numPy files (see prompt. search for npz)

# Filter out observations where metabolites are only consumed by a few microbes

# At the genus, class, order, family levels the precision recall is X

# label metabolites as everybody likes vs discriminatory

# See if the flip of the model is valid


In [31]:
# test_microbiome_experiment.py
"""Prototype: score a microbiome experiment against learned metabolite preferences."""

import pandas as pd
from pathlib import Path

from pref import map_compound_name, build_exometabolite
from model import load_model, BPRModel
from config import CACHE_DIR

# =============================================================================
# Load model
# =============================================================================

consumed_model = load_model(Path(CACHE_DIR) / "bpr_consumed", model_class=BPRModel)

# =============================================================================
# Load and map exudate compounds
# =============================================================================

exu_compounds = pd.read_csv(
    "~/Downloads/RingTrial_WoM_style_data_Dunnet.xlsx - Sheet1.tsv", sep="\t"
)
syncom_phylogeny = pd.read_csv("~/Downloads/phylogeny - SynComs.tsv", sep="\t")
syncom_phylogeny = syncom_phylogeny[
    syncom_phylogeny["SynCom Name"].str.contains("SynCom1")
]

exu_compounds = exu_compounds.iloc[0]
exu_compounds = exu_compounds[2:]

mapped_names = []
for compound in exu_compounds.index:
    mapped = map_compound_name(compound)

    if mapped["representative_compound"] is None:
        print(f"Unmatched: Compound {compound} could not be mapped")
        continue
    elif mapped["representative_compound"] not in consumed_model.metabolite_to_idx:
        print(
            f"Not in model: Compound {compound} mapped to "
            f"{mapped['representative_compound']} which is not in model"
        )
        continue
    else:
        mapped_names.append(mapped["representative_compound"])

# =============================================================================
# Predict top genera for each compound
# =============================================================================

out = []
for compound in mapped_names:
    results = consumed_model.predict_for_taxon_all_members(
        compound, level="genus"
    ) if hasattr(consumed_model, "predict_for_taxon_all_members") else []

    # Use taxonomy embeddings to score each genus on this compound
    if consumed_model.taxonomy_embeddings and "genus" in consumed_model.taxonomy_embeddings:
        c_idx = consumed_model.metabolite_to_idx[compound]
        met_emb = consumed_model.model.metabolite_embeddings[c_idx]

        for taxon, tax_emb in consumed_model.taxonomy_embeddings["genus"].items():
            score = float(tax_emb @ met_emb)
            out.append({"compound": compound, "taxon": taxon, "score": score})

out_df = pd.DataFrame(out)
out_df = pd.pivot_table(
    out_df, index="taxon", columns="compound", values="score", fill_value=0
)

# =============================================================================
# Load WoM assertions for comparison
# =============================================================================

wom_assertions = build_exometabolite()
cols = [
    "representative_compound",
    "organism_id",
    "assertion_value",
    "assertion",
    "kingdom",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
]
wom_assertions = wom_assertions[cols]

# =============================================================================
# Score all isolates on all metabolites
# =============================================================================

all_scores = []
for isolate in consumed_model.microbe_to_idx:
    predictions = consumed_model.predict_for_microbe(isolate)
    scores = pd.Series(
        {name: score for name, score in predictions},
        name=isolate,
    )
    all_scores.append(scores)

out_df = pd.concat(all_scores, axis=1).T
out_df.index.name = "organism_id"
out_df = out_df.reset_index().melt(
    id_vars="organism_id",
    var_name="representative_compound",
    value_name="score",
)

# Merge with assertions
merge_cols = ["organism_id", "representative_compound"]
out_df = pd.merge(
    out_df,
    wom_assertions[merge_cols + ["assertion_value", "assertion"]],
    on=merge_cols,
    how="left",
)

# Merge taxonomy
taxon_df = wom_assertions[
    ["organism_id", "kingdom", "phylum", "class", "order", "family", "genus"]
].drop_duplicates()
out_df = pd.merge(out_df, taxon_df, on="organism_id", how="left")

print(out_df.sample(30))
out_df.to_csv("exometabolite_predictions_and_measurements.tsv", sep="\t", index=False)
print(out_df.head())

Not in model: Compound 4-hydroxy-2-quinolinecarboxylic acid mapped to kynurenic acid which is not in model
Not in model: Compound 4-imidazoleacetic acid mapped to imidazol-4-ylacetic acid which is not in model
Not in model: Compound 4-quinolinecarboxylic acid mapped to quinoline-4-carboxylic acid which is not in model
Not in model: Compound N-trimethyllysine mapped to n(6),n(6),n(6)-trimethyl-l-lysine which is not in model
Not in model: Compound agmatine mapped to agmatine which is not in model
Not in model: Compound allantoin mapped to allantoin which is not in model
Not in model: Compound cysteic acid mapped to cysteic acid which is not in model
Not in model: Compound dopamine mapped to dopamine which is not in model
Unmatched: Compound hexosaminic acid could not be mapped
Not in model: Compound histamine mapped to histamine which is not in model
Not in model: Compound histidinol mapped to histidinol which is not in model
Not in model: Compound putrescine mapped to putrescine which i

In [40]:
# idx1 = out_df['representative_compound'].isin(mapped_names)
# idx2 = out_df['organism_id'].isin(syncom_phylogeny['Isolate'])
# out_df = out_df[idx1 & idx2]
p = pd.pivot_table(out_df, columns="organism_id", index="representative_compound", values="score", fill_value=0)

# for each column convert to rank and scale to 0-1
p = p.rank(axis=0, method='min', ascending=False)
# p = p / p.max()
p = p[p.index.isin(mapped_names)]
cols = [c for c in p.columns if c in syncom_phylogeny['Isolate'].values]
p = p[cols]
out = []
for c in p.columns:
    # find top 10 scores    top_10 = p[c].nlargest(10)
    top_10 = p[c].nlargest(15)
    # print(f"Top 10 for {c}:",top_10.mean())
    out.append({"isolate": c, "mean_top_10_score": top_10.mean()})
out = pd.DataFrame(out)
# out = pd.merge(out , syncom_phylogeny, left_on="isolate", right_on="Isolate")
out.sort_values("mean_top_10_score", ascending=False)

,isolate,mean_top_10_score
2,BoseaOAE506,86.733333
14,VariovoraxOAS795,78.733333
9,MucilaginibacterOAE612,78.133333
11,PaenibacillusOAE614,77.000000
13,RhodococcusOAS809,76.066667
6,LysobacterOAE881,75.333333
8,MethylobacteriumOAE515,74.733333
5,ChitinophagaOAE865,74.466667
4,BurkholderiaOAS925,73.800000
10,NiastellaOAS944,72.866667


In [1]:
new_taxonomy_filename = './data/oriented_ENIGMA_MCAFES_tax.txt'
new_taxonomy = pd.read_csv(new_taxonomy_filename, sep='\t')
# since it has no column headers, add them
new_taxonomy.columns = ['organism_id', 'scores','nothing','taxonomy']
new_taxonomy['organism_id'] = new_taxonomy['organism_id'].str.replace(' <unknown description>','').str.strip()
new_taxonomy = new_taxonomy[new_taxonomy['organism_id'].isin(wrapper.idx_to_microbe.values())]
new_taxonomy.set_index('organism_id', inplace=True)
new_taxonomy.drop(columns=['taxonomy','nothing'], inplace=True)
temp = []
for org_id, row in new_taxonomy.iterrows():
    
    scores = row['scores'].split(',')
    for score in scores:
        
        parts = score.split(':')
        taxonomic_level = parts[0]
        v = parts[1]
        parts = v.split('(')
        assignment = parts[0]
        score = parts[1].replace(')','')
        d = {'organism_id': org_id,'taxonomic_level': taxonomic_level, 'assignment': assignment, 'score': float(score)}
        temp.append(d)
new_taxonomy_expanded = pd.DataFrame(temp)
level_map = {'f':'family', 'g':'genus', 'o':'order', 'c':'class', 'p':'phylum','s':'species'}
new_taxonomy_expanded['taxonomic_level'] = new_taxonomy_expanded['taxonomic_level'].map(level_map)
new_taxonomy_expanded = new_taxonomy_expanded[new_taxonomy_expanded['score'] >= 0]
new_taxonomy_expanded = new_taxonomy_expanded.pivot(index='organism_id', columns='taxonomic_level', values='assignment')
models = load_bpr_models()
wrapper = models['consumed']
embeddings = {}
for org in unique_organisms:
    if org not in wrapper.idx_to_microbe.values():
        print(f"Organism {org} not in model, skipping")
        continue
    pos = wrapper.get_microbe_embedding(org)
    embeddings[org] = pos
embeddings = pd.DataFrame.from_dict(embeddings, orient='index')
embeddings.index.name = 'organism_id'
# calculate all by all distances

dist_matrix = pairwise_distances(embeddings.values, metric='cosine')
dist_df = pd.DataFrame(dist_matrix, index=embeddings.index, columns=embeddings.index)
dist_df.index.name = 'organism_id_1'
dist_df = dist_df.melt(ignore_index=False, var_name='organism_id_2', value_name='embedding_distance')
dist_df.reset_index(inplace=True)
cols = ['organism_id_1', 'organism_id_2']
# plot
sorted_names = np.sort(dist_df[cols],axis=1)
dist_df[cols] = sorted_names
dist_df = dist_df.drop_duplicates(subset=cols)
dist_df = dist_df[dist_df['organism_id_1'] != dist_df['organism_id_2']]
merged = dist_df.merge(new_taxonomy_expanded, left_on='organism_id_1', right_index=True)
merged = merged.merge(new_taxonomy_expanded, left_on='organism_id_2', right_index=True, suffixes=('_1','_2'))
subset = merged[merged['genus_1'] == merged['genus_2']].copy()
not_same = merged[merged['genus_1'] != merged['genus_2']].copy()
fig,ax = plt.subplots(1,1, figsize=(7,3))
edges = np.linspace(merged['embedding_distance'].min(), merged['embedding_distance'].max(), 51)
ax.hist(subset['embedding_distance'], bins=edges, alpha=0.7, label='same genus',density=True)
# ax.hist(merged['embedding_distance'], bins=edges, alpha=0.7, label='all pairs',density=True)
ax.hist(not_same['embedding_distance'], bins=edges, alpha=0.7, label='different genus',density=True)
ax.set_xlabel('Embedding distance (cosine)')
ax.set_ylabel('Density')
ax.legend()

NameError: name 'pd' is not defined

In [ ]:
# import pandas as pd
# import json
# from sklearn.metrics import pairwise_distances
# import numpy as np
# import matplotlib.pyplot as plt
# from model import load_bpr_models

# phylo_dist_filname = './data/phylogenetic_distances.json'
# # this is a biopython tree object saved as json
# phylo_dist = json.load(open(phylo_dist_filname, 'r'))
# # this has the all by all distances as key and dict of all distances as value
# phylo_dist = phylo_dist['distances']
# # lets convert to a melted dataframe
# phylo_dist = [(k1,k2,v) for k1,vv in phylo_dist.items() for k2,v in vv.items()]
# phylo_dist = pd.DataFrame(phylo_dist, columns=['organism_id_1', 'organism_id_2', 'phylo_distance'])
# unique_organisms = set(phylo_dist['organism_id_1'].unique().tolist() + phylo_dist['organism_id_2'].unique().tolist())
# # get the embeddings for each organism
# models = load_bpr_models()
# wrapper = models['consumed']
# embeddings = {}
# for org in unique_organisms:
#     if org not in wrapper.idx_to_microbe.values():
#         print(f"Organism {org} not in model, skipping")
#         continue
#     pos = wrapper.get_microbe_embedding(org)
#     embeddings[org] = pos
# embeddings = pd.DataFrame.from_dict(embeddings, orient='index')
# embeddings.index.name = 'organism_id'
# # calculate all by all distances

# dist_matrix = pairwise_distances(embeddings.values, metric='cosine')
# dist_df = pd.DataFrame(dist_matrix, index=embeddings.index, columns=embeddings.index)
# dist_df.index.name = 'organism_id_1'
# dist_df = dist_df.melt(ignore_index=False, var_name='organism_id_2', value_name='embedding_distance')
# dist_df.reset_index(inplace=True)
# # merge with phylo distances
# cols = ['organism_id_1', 'organism_id_2']
# merged = pd.merge(phylo_dist, dist_df, on=cols, how='inner')
# # plot
# sorted_names = np.sort(merged[cols],axis=1)
# merged[cols] = sorted_names
# merged = merged.drop_duplicates(subset=cols)
# merged = merged[merged['organism_id_1'] != merged['organism_id_2']]

# subset = merged.copy()
# subset.sort_values('phylo_distance', inplace=True,ascending=True)
# subset.drop_duplicates('organism_id_1', inplace=True)

# fig,ax = plt.subplots(1,1, figsize=(4,4))
# ax.plot(merged['phylo_distance'], merged['embedding_distance'], '.', alpha=0.7, label='all pairs')
# ax.plot(subset['phylo_distance'], subset['embedding_distance'], '.',label='nearest relative')

# ax.legend(frameon=False)
# ax.set_xlabel('Phylogenetic Distance')
# ax.set_ylabel('Embedding Distance (cosine)')

In [1]:
from model import load_model
from predict import MicrobePredictions, CompoundPredictions

models = load_bpr_models()
wrapper = models['consumed']

# Test MicrobePredictions
mp = MicrobePredictions(wrapper)
mp.rank(['glutamate(1-)', 'serine', 'guanine'])
mp.extrapolate(to='genus')
print(mp.summarize(top_k=10))

# Test CompoundPredictions  
cp = CompoundPredictions(wrapper)
cp.rank([
    {'isolate': 'BurkholderiaOAS925'},
    {'genus': 'Arthrobacter'},
])
cp.extrapolate()
print(cp.summarize(top_k=10))

ImportError: cannot import name 'BPRModelWrapper' from 'model' (/Users/bpb/repos/webofmicrobes/wom/model.py)

In [ ]:
mp = MicrobePredictions(wrapper)
mp.rank(['glutamate', 'L-serine', 'guanin'])  # intentional variations
print(mp.compound_mapping)  # see what got mapped

ImportError: cannot import name 'predict_taxa_for_compound' from 'model' (/Users/bpb/repos/webofmicrobes/wom/model.py)

In [ ]:
out_df

In [ ]:
from pref import build_exometabolite
wom_assertions = build_exometabolite()
cols = ['representative_compound','organism_id','assertion_value','assertion','kingdom','phylum','class','order','family','genus']
wom_assertions = wom_assertions[cols]
wom_assertions

In [3]:
all_scores = []
for isolate in consumed_model.microbe_to_idx:
    scores = check_microbe_preferences(consumed_model, isolate, [], return_all_scores=True)
    scores = pd.Series(scores)
    scores = scores.to_frame()
    # scores['rank'] = scores.rank(method='min')
    scores.columns = [isolate]
    all_scores.append(scores)
out_df = pd.concat(all_scores, axis=1).T
# melt out_df use index and columns as new columns. call the values 'score'
out_df.index.name= 'organism_id'
out_df = out_df.reset_index().melt(id_vars='organism_id', var_name='representative_compound', value_name='score')
merge_cols = ['organism_id', 'representative_compound']
out_df = pd.merge(out_df, wom_assertions[merge_cols + ['assertion_value','assertion']], left_on=merge_cols, right_on=merge_cols, how='left')
taxon_df = wom_assertions[['organism_id','kingdom','phylum','class','order','family','genus']].drop_duplicates()
out_df = pd.merge(out_df, taxon_df, left_on='organism_id', right_on='organism_id', how='left')

out_df.sample(30)
out_df.to_csv('exometabolite_predictions_and_measurements.tsv', sep='\t', index=False)
out_df.head()

NameError: name 'consumed_model' is not defined

In [ ]:
# for each representative_compound I want to know how many organism_id are there where the assertion is consumed
g = out_df[out_df['assertion']=='consumed'].groupby('representative_compound')['organism_id'].nunique().sort_values(ascending=False)


In [ ]:
g = pd.pivot_table(out_df, index='organism_id', columns='representative_compound', values='score', fill_value=0.0)  
g = g.astype('float32')
# keep_cols = g.max()>5
# g = g.loc[:,keep_cols]
import seaborn as sns
sns.clustermap(g, figsize=(20,20), method='ward')#,vmin=0, vmax=10)

In [ ]:
# what cutoff score would indicate a likely consumption event?
g = out_df[out_df['assertion'] == 'consumed'].copy()
g = g.sort_values('score', ascending=False)
g.drop_duplicates('organism_id', inplace=True)
fig,ax = plt.subplots(figsize=(10,5))
edges = np.linspace(0, out_df['score'].max(), 50)
ax.hist(g['score'], bins=edges, alpha=0.7,label='Asserted consumed compounds')
temp = out_df.sort_values('score',ascending=False).drop_duplicates('organism_id').copy()
ax.hist(temp['score'], bins=edges, alpha=0.3,label='All compounds')
ax.set_xlabel('Top consumption score for asserted consumed compounds')
ax.set_ylabel('Number of organisms')
ax.legend()
plt.show()


In [ ]:
g = [d.sort_values('score', ascending=False).head(5) for _,d in out_df.groupby('organism_id')]
x = [gg.iloc[0]['score'] for gg in g if len(gg) > 0]
y = [gg.iloc[-1]['score'] for gg in g if len(gg) > 0]
fig,ax = plt.subplots(1,1, figsize=(7,7))

ax.plot(x,y,'.')
ax.set_xlabel('Top predicted score')
ax.set_ylabel('5th predicted score')

In [ ]:
out_df

In [ ]:
sns.clustermap(out_df.pivot(index='organism_id', columns='representative_compound', values='score').fillna(0), figsize=(10,10))

In [ ]:
fig,ax = plt.subplots(figsize=(10,5))
import seaborn as sns
temp = out_df.sort_values('score', ascending=False).drop_duplicates('organism_id')
sns.boxplot(data=temp, x='assertion', y='score', ax=ax)
sns.swarmplot(data=temp, x='assertion', y='score', color='black', size=10, ax=ax)

In [ ]:
idx1 = out_df['score'] > 3
idx2 = pd.isna(out_df['assertion_value'])
idx = idx1 & idx2
print(out_df.loc[idx,'organism_id'].value_counts().to_string())
out_df[idx].sort_values(['class','order','family','genus','score'])

In [ ]:
idx1 = out_df['assertion']=='consumed'
idx2 = pd.notna(out_df['assertion'])
idx = idx1 & idx2

fig,ax = plt.subplots(figsize=(8,6))
edges = np.linspace(out_df['score'].min(), out_df['score'].max(), 50)
out_df.loc[idx,'score'].hist(bins=edges, alpha=0.5,density=False,label='measured consumed', ax=ax)
# out_df.loc[~idx,'score'].hist(bins=50, alpha=0.5,density=True,label='not measured consumed', ax=ax)
out_df.loc[out_df['assertion'].isna(),'score'].hist(bins=edges, alpha=0.5,density=False,label='no measurement', ax=ax)
ax.set_xlabel('BSPR predicted consumption score')
ax.set_ylabel('Number of observations')
ax.legend()

In [ ]:
out = []
for isolate in syncom_phylogeny['Isolate'].unique():
    if isolate not in consumed_model.microbe_to_idx:
        print(f"Isolate {isolate} not in model")
        continue
    scores = check_microbe_preferences(consumed_model, isolate, [], return_all_scores=True)
    scores = pd.Series(scores)
    # replace scores with ranks
    scores = scores.rank(ascending=False)
    # normalize to 0-1
    # scores = scores - scores.min()
    # scores = scores / scores.max()
    scores = scores.to_frame()
    scores['isolate'] = isolate
    scores.index.name = 'compound'
    scores.columns = ['score', 'isolate']

    scores.reset_index(inplace=True)
    scores['in_media'] = scores['compound'].isin(mapped_names)
    out.append(scores)
out = pd.concat(out, ignore_index=True)
out = pd.pivot_table(out, index='compound',columns='isolate', values='score')
cols = out.columns.tolist()
out.sort_values(by=cols, ascending=False, inplace=True)
out.reset_index(inplace=True)
out['in_media'] = out['compound'].isin(mapped_names)
out.head()


In [ ]:
out_df

In [ ]:
# After your pivot, before normalization
# out_df should have isolates as columns, compounds as rows

diagnostics = []
temp = out_df.T.copy()
for isolate in temp.columns:
    scores = temp[isolate].values
    diagnostics.append({
        'isolate': isolate,
        'raw_sum': scores.sum(),
        'min': scores.min(),
        'max': scores.max(),
        'range': scores.max() - scores.min(),
        'mean': scores.mean(),
        'std': scores.std(),
        'normalized_sum': ((scores - scores.min()) / (scores.max() - scores.min() + 1e-8)).sum()
    })

diag_df = pd.DataFrame(diagnostics).sort_values('normalized_sum')
print(diag_df.to_string())

In [ ]:
out.sort_values('BurkholderiaOAS925',ascending=True,inplace=True)
out_media = out[out['in_media']].copy()
# out_media = out.copy()
out_media.drop(columns=['in_media'], inplace=True)
out_media.set_index('compound', inplace=True)
# redo the ranking
# out_media = out_media.rank(ascending=False)
out_media.sort_values('BrevibacillusOAP136',ascending=True,inplace=True)
out_media = out_media - out_media.min()
out_media = out_media / out_media.max()
# out_media.sum().sort_values()
out_media['geometric_mean'] = out_media.prod(axis=1)**(1/out_media.shape[1])
out_media.sort_values('geometric_mean', ascending=True, inplace=True)
out_media.drop(columns=['geometric_mean'], inplace=True)
# locate colorbar away from plot
sns.clustermap(out_media.head(20).T, 
               cmap='viridis', 
               cbar_kws={'label':'Predicted Rank\n(lower is better)',
                           'location':'left'}, 
               figsize=(10,6),
               vmin=0,
               vmax=50)
out_media.sum().sort_values()

In [ ]:
fig,ax = plt.subplots(figsize=(17,7))
import numpy as np
# edges = np.linspace(out_media.values.min(), out_media.values.max(), 10)
edges = np.linspace(0,1,10)

out_media.head(50).hist(ax=ax, bins=edges,sharex=True,sharey=True)

plt.tight_layout()

In [ ]:
fig,ax = plt.subplots(figsize=(17,7))
import numpy as np
# edges = np.linspace(out_media.values.min(), out_media.values.max(), 10)
edges = np.linspace(-10,10,20)
out_media.head(50).hist(ax=ax, bins=edges,sharex=True,sharey=True)

plt.tight_layout()

In [ ]:
# import matplotlib.pyplot as plt
# fix,ax = plt.subplots(figsize=(7,20),nrows = len(out), sharex=True)
# for i,scores in enumerate(out):
#     scores = scores.sort_values('score', ascending=False)#.head(50)
#     scores.set_index('compound', inplace=True)
#     scores['score'].plot.bar(ax=ax[i])
#     ax[i].set_title(scores['isolate'].iloc[0], fontsize=8)
#     # for j,bar in enumerate(ax[i].containers[0]):
#         # if scores['in_media'].iloc[j]:
#             # bar.set_color('orange')
# plt.tight_layout()

import seaborn as sns
import matplotlib.pyplot as plt
fig,ax = plt.subplots(figsize=(10,6))
d = out.melt(id_vars=['compound','in_media'],var_name='isolate', value_name='score')
sns.boxplot(data=d, x='isolate', y='score', hue='in_media',  ax=ax,showmeans=True)
sns.swarmplot(data=d, x='isolate', y='score', hue='in_media', ax=ax, dodge=True)
plt.xticks(rotation=90)
plt.tight_layout()
# ax.set_ylim(0.4,0.5)
plt.show()

In [ ]:
scores

In [ ]:
out = score_microbes_on_medium(consumed_model, mapped_names)
out = out.to_frame()
out.index.name = 'Isolate'
out.columns = ['Score']
out.reset_index(inplace=True)
cols = ['Isolate', 'Phylum','Class','Order','Family', 'Genus']
out = pd.merge(out, syncom_phylogeny[cols], left_on='Isolate', right_on='Isolate')
out.drop_duplicates(inplace=True)
out.sort_values('Score', ascending=False, inplace=True)
print(out.to_string())

In [ ]:
from wom import consumed_by
out = []
for genus in syncom_phylogeny['Genus']:
    temp = consumed_by(genus=genus,top_k=50)
    temp['Genus'] = genus
    out.append(temp)
out_df = pd.concat(out)
out_df = pd.pivot_table(out_df, index='Genus', columns='chebi_id', values='mean_score', fill_value=0)
cols = out_df.columns.intersection(mapped_names)
out_df = out_df[cols]
import seaborn as sns
sns.clustermap(out_df, cmap='viridis', figsize=(10,6))

In [ ]:
(out_df>5).sum(axis=1).sort_values()

In [ ]:
# from pref import map_compound_name
# from model import predict_taxa_for_compound, predict_taxa_for_mixture
# import pandas as pd
# from model import load_bpr_models

# models = load_bpr_models()
# consumed_model = models['consumed']

# exu_compounds = pd.read_csv('~/Downloads/RingTrial_WoM_style_data_Dunnet.xlsx - Sheet1.tsv', sep='\t')
# syncom_phylogeny = pd.read_csv('~/Downloads/phylogeny - SynComs.tsv', sep='\t')
# syncom_phylogeny = syncom_phylogeny[syncom_phylogeny['SynCom Name'].str.contains('SynCom1')]
# exu_compounds = exu_compounds.iloc[0]
# exu_compounds = exu_compounds[2:]
# mapped_names = []
# for compound in exu_compounds.index:
#     mapped = map_compound_name(compound)

#     if mapped['representative_compound'] is None:
#         print(f"Unmatched: Compound {compound} could not be mapped")
#         continue
#     elif mapped['representative_compound'] not in consumed_model.metabolite_to_idx:
#         print(f"Not in model: Compound {compound} mapped to {mapped['representative_compound']} which is not in model")
#         continue
#     else:
#         mapped_names.append(mapped['representative_compound'])
        
#     # print(compound, '->', mapped)
# models = load_bpr_models()

# Simple mixture
# top_genera = predict_taxa_for_mixture(consumed, mapped_names, rank="genus")

# # Weighted by concentration
# mixture = ['glucose', 'acetate', 'lactate']
# concentrations = [10.0, 5.0, 2.0]  # mM or arbitrary units
# top_genera = predict_taxa_for_mixture(consumed, mixture, weights=concentrations)

# # Find taxa that tolerate all components (no strong dislikes)
# top_genera = predict_taxa_for_mixture(consumed, mixture, method="min")
# top_genera

In [ ]:
consumed = models['consumed']

top_genera = predict_taxa_for_mixture(consumed, mapped_names, rank="genus", top_k=2000,method='geometric')
top_genera = pd.DataFrame(top_genera, columns=['Genus', 'Score'])
top_genera = top_genera[top_genera['Genus'].isin(syncom_phylogeny['Genus'])]
import matplotlib.pyplot as plt
fig,ax = plt.subplots(figsize=(7,3))
top_genera.set_index('Genus').sort_values('Score').plot.barh(ax=ax)

In [ ]:
# def score_microbes_on_medium(
#     wrapper: BPRModelWrapper,
#     compound_list: list[str],
#     normalize: bool = True


In [ ]:
from model import check_microbe_preferences
out = []
for isolate in syncom_phylogeny['Isolate']:
    if isolate not in consumed_model.microbe_to_idx.keys():
        print(f"Isolate {isolate} not in model, skipping")
        continue
    result = check_microbe_preferences(consumed_model, isolate, mapped_names, top_k=1000)
    temp = pd.DataFrame(result['compound_rankings']).T
    temp['isolate'] = isolate
    out.append(temp)
out_df = pd.concat(out)
out_df.index.name = 'compound'
out_df.reset_index(inplace=True)
out_df = pd.pivot_table(out_df, index='isolate', columns='compound', values='rank', fill_value=0)
# out_df = out_df.T
# out_df = out_df.subtract(out_df.min(axis=0), axis=1)
# out_df = out_df.divide(out_df.max(axis=0), axis=1)

# out_df.mean().sort_values()
out_df

In [ ]:
# USE THE 5 MEMBER SYNCOM DATA FROM 2 YEARS AGO TO VALIDATE THESE PREDICTIONS
# SEE IF THESE RANKINGS CORRELATE WITH GROWITH RATES OF ISOLATES

In [ ]:
for c in out_df.columns:
    idx = out_df[c].idxmin()
    # best_isolate = idx
    print(idx, c, out_df[c].min(), out_df[c].max())

In [ ]:
import pandas as pd

# Assuming df is your dataframe with compounds as rows, isolates as columns

def analyze_rank_distribution(df):
    results = []
    
    for isolate in df.columns:
        ranks = df[isolate].values
        
        results.append({
            'isolate': isolate,
            'top_5': (ranks <= 5).sum(),
            'top_10': (ranks <= 10).sum(),
            'top_20': (ranks <= 20).sum(),
            'top_50': (ranks <= 50).sum(),
            'bottom_50': (ranks >= 170).sum(),  # ~220 total compounds
            'raw_sum': ranks.sum(),
            'min': ranks.min(),
            'max': ranks.max(),
        })
    
    return pd.DataFrame(results).sort_values('top_10', ascending=False)

t = analyze_rank_distribution(out_df.T)
print(t.to_string())


In [ ]:
def analyze_middle_ranks(df):
    results = []
    
    for isolate in df.columns:
        ranks = df[isolate].values
        
        # Count compounds in different tiers
        top_tier = (ranks <= 20).sum()
        mid_tier = ((ranks > 20) & (ranks <= 100)).sum()
        low_tier = (ranks > 100).sum()
        
        # Variance in preferences
        std = ranks.std()
        
        results.append({
            'isolate': isolate,
            'top_tier': top_tier,    # strong preference
            'mid_tier': mid_tier,    # moderate preference  
            'low_tier': low_tier,    # weak/negative preference
            'std': std,
            'raw_sum': ranks.sum(),
        })
    
    return pd.DataFrame(results).sort_values('mid_tier')

t = analyze_middle_ranks(out_df.T)
print(t.to_string())

In [ ]:
nucleotides = ['guanine', 'cytidine', 'uridine', 'guanosine', 'cytosine', 
               "2'-deoxycytidine"]
temp = out_df.T.copy()
nuc_ranks = temp.loc[temp.index.isin(nucleotides)]
nuc_sum = nuc_ranks.sum().sort_values()
print(nuc_sum)

In [ ]:
amino_acids = ['glutamate(1-)', 'aspartic acid', 'asparagine', 'glutamine', 
               'serine', 'arginine', 'lysine', 'histidine', 'proline',
               'tyrosine', 'tryptophan', 'cystine']
temp = out_df.T.copy()
aa_ranks = temp.loc[temp.index.isin(amino_acids)]
aa_sum = aa_ranks.sum().sort_values()
print(aa_sum)
# aa_ranks

In [ ]:
print(out_df.T.astype(int).to_string())

In [ ]:
# After your pivot, before normalization
# out_df should have isolates as columns, compounds as rows

diagnostics = []
temp = out_df.T.copy()
for isolate in temp.columns:
    scores = temp[isolate].values
    diagnostics.append({
        'isolate': isolate,
        'raw_sum': scores.sum(),
        'min': scores.min(),
        'max': scores.max(),
        'range': scores.max() - scores.min(),
        'mean': scores.mean(),
        'std': scores.std(),
        'normalized_sum': ((scores - scores.min()) / (scores.max() - scores.min() + 1e-8)).sum()
    })

diag_df = pd.DataFrame(diagnostics).sort_values('normalized_sum')
print(diag_df.to_string())

In [ ]:
# for each microbe make a bar chart of all the compounds it is predicted to consume
# color the bars if they are in the media
# rerank predictions based on media composition

In [ ]:
out_df.sum().sort_values()

In [ ]:
from model import check_microbe_preferences

out = []
for isolate in syncom_phylogeny['Isolate'].unique():
    if isolate not in consumed_model.microbe_to_idx.keys():
        print(f"Isolate {isolate} not in model, skipping")
        continue
    result = check_microbe_preferences(consumed_model, isolate, mapped_names, top_k=5000)
    # print(f"Microbe: {isolate}")
    # print(f"  Overlap in top predictions: {result['overlap_count']}")
    percentile_sum = 0
    for compound in result['overlap_in_top']:
        rank_info = result['compound_rankings'][compound]
        percentile_sum += rank_info['percentile']
        out.append((isolate, compound, rank_info['rank']))
out = pd.DataFrame(out, columns=['Isolate', 'Compound', 'Rank'])
out = out.pivot(index='Isolate', columns='Compound', values='Rank')
good_cols = (out<20).sum(axis=0)>0
out = out.loc[:,good_cols]
import seaborn as sns
sns.clustermap(out, cmap='viridis', figsize=(10,6),vmin=0,vmax=20)

In [ ]:
(out<20).sum(axis=1).sort_values()

In [ ]:
out_df[out_df.index.isin(syncom_phylogeny['Genus'])].sum(axis=1).sort_values(ascending=False).head(20)

In [ ]:
temp = out_df[out_df.index.isin(syncom_phylogeny['Genus'])].copy()
import seaborn as sns
sns.clustermap(temp.transpose(), cmap='viridis', figsize=(10,15))

In [ ]:
out_df.sum(axis=1).sort_values(ascending=False).head(20)

In [ ]:
import wom
	# Bacteria; Pseudomonadota; Alphaproteobacteria; Candidatus Pelagibacterales; Candidatus Pelagibacteraceae; Candidatus Pelagibacter; Candidatus Pelagibacter communis
# The provided text describes the taxonomic classification of a group of abundant marine bacteria, 
# belonging to the genus Candidatus Pelagibacter within the SAR11 clade 
# (family Pelagibacteraceae, order Pelagibacterales, class Alphaproteobacteria, phylum Pseudomonadota), specifically
# Query for a genus we haven't measured
# result = wom.predict_consumption_novel(
    # taxonomy={"genus": "Pseudomonas", "family": "Pseudomonadaceae"}
# )
# result = wom.predict_consumption_novel(
#     taxonomy={ "family": "Pelagibacteraceae"}
# )
# print(result)

# Get explanation of the prediction
context = wom.get_phylogenetic_context(
    taxonomy={"genus": "Gordonia"},
    model_type="consumed"
)
print(f"Based on {len(context['neighbors'])} neighbors")
print(f"Confidence: {context['confidence']:.2f}")
for n in context["neighbors"][:3]:
    print(f"  {n['taxon']} (distance={n['distance']}, weight={n['weight']:.2f})")

In [ ]:
#  ['Cysteine / Cystine (dimer of Cysteine)', 'Cystine (dimer of Cysteine)', 'Cystine / Cysteine', 'N-Acetyl-D-Muramic acid ']
from pref import lookup_chebi_by_name
print(lookup_chebi_by_name("L-cystine"))
print(lookup_chebi_by_name("N-acetyl-D-muramic acid"))

In [ ]:
from pref import load_chebi

chebi_df = load_chebi()
matches = chebi_df[chebi_df["name"].str.contains("muramic", case=False, na=False)]
print(matches[["compound_id", "name", "name_norm"]].drop_duplicates("name").to_string())

In [ ]:
# %pip install rapidfuzz
# %pip install implicit
# 

In [ ]:
from pref import _load_and_melt_exometabolite, _process_exometabolite
from config import DATA_DIR
from pathlib import Path
import os
file = os.path.join(DATA_DIR, "NLDM_syncom.tsv")
if isinstance(file, str):
    file = Path(file)
df = _load_and_melt_exometabolite(file)
df_processed = _process_exometabolite(df)

file = os.path.join(DATA_DIR, "Biggs_isolates.tsv")
if isinstance(file, str):
    file = Path(file)
df_training_data = _load_and_melt_exometabolite(file)
df_training_data = _process_exometabolite(df_training_data)
df_processed

In [ ]:
df_val_taxa = df_processed[["genus", "family"]].drop_duplicates().reset_index(drop=True)
df_training_taxa = df_training_data[["genus", "family"]].drop_duplicates().reset_index(drop=True)
df_training_taxa['training'] = True
common_taxa = pd.merge(
    df_val_taxa,
    df_training_taxa,
    on=["genus", "family"],
    how="left")
common_taxa

In [ ]:
taxa_df = df_processed[["genus", "family"]].drop_duplicates().reset_index(drop=True)
for idx, row in taxa_df.iterrows():
    genus = row["genus"]
    family = row["family"]
    print(f"Predicting for genus={genus}, family={family}")
    df_predict = wom.predict_consumption_novel(taxonomy={"genus": genus, "family": family},top_k=100)
    df_predict = df_predict[df_predict['score']>0]
    df_actual = df_processed[df_processed["genus"] == genus].copy()
    df_actual = df_actual[df_actual["assertion_value"] < -2]
    tp = df_actual['chebi_name'].unique().tolist()
    predicted = df_predict['chebi_name'].unique().tolist()
    print(f"Actual consumed metabolites ({len(tp)}): {tp}")
    print(f"Predicted consumed metabolites ({len(predicted)}): {predicted}")
    print(f"True positives ({len(set(tp).intersection(set(predicted)))}): {set(tp).intersection(set(predicted))}")
    print(f"False positives ({len(set(predicted)-set(tp))}): {set(predicted)-set(tp)}")
    print(f"False negatives ({len(set(tp)-set(predicted))}): {set(tp)-set(predicted)}")
    print("-----")


In [ ]:
data = []
taxa_df = df_processed[["genus", "family"]].drop_duplicates().reset_index(drop=True)
for idx, row in taxa_df.iterrows():
    genus = row["genus"]
    family = row["family"]
    temp = wom.consumed_by(genus=genus, top_k=1000)
    temp = temp[temp['mean_score']>0]
    temp['consumer_genus'] = genus
    data.append(temp)
import pandas as pd
df = pd.concat(data)
df = pd.pivot_table(df, index='chebi_name', columns='consumer_genus', values='mean_score', fill_value=0)
# idx = df.max(axis=1)>5
# df = df.loc[idx]
import seaborn as sns
import matplotlib.pyplot as plt

h = sns.clustermap(df, cmap='viridis')


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Create your predicted clustermap first
h = sns.clustermap(df, cmap='viridis')

# Extract the clustered order
row_order = h.dendrogram_row.reordered_ind
col_order = h.dendrogram_col.reordered_ind

# Get the ordered labels
ordered_metabolites = df.index[row_order]
ordered_genera = df.columns[col_order]

# Get figure size
fig_size = h.fig.get_size_inches()

plt.close()  # Close the initial clustermap

# Build the actual data matrix
actual_data = []
for genus in df.columns:
    df_actual = df_processed[df_processed["genus"] == genus].copy()
    df_actual = df_actual[df_actual["assertion_value"] < -2]
    # Assuming there's a column for metabolite name - adjust as needed
    for _, row in df_actual.iterrows():
        actual_data.append({
            'chebi_name': row['chebi_name'],  # adjust column name if different
            'consumer_genus': genus,
            'value': row['assertion_value']  # or use abs() if you want positive values
        })

df_actual_all = pd.DataFrame(actual_data)
df_actual_pivot = pd.pivot_table(
    df_actual_all, 
    index='chebi_name', 
    columns='consumer_genus', 
    values='value', 
    fill_value=0
)

# Reindex to match the predicted data (fill missing with 0)
df_actual_pivot = df_actual_pivot.reindex(
    index=ordered_metabolites, 
    columns=ordered_genera, 
    fill_value=0
)

# Reorder predicted data to match
df_predicted_ordered = df.loc[ordered_metabolites, ordered_genera]

# Create side-by-side figure
fig, axes = plt.subplots(1, 2, figsize=(fig_size[0] * 1.8, fig_size[1]))

sns.heatmap(df_predicted_ordered, cmap='viridis', ax=axes[0], 
            xticklabels=True, yticklabels=True)
axes[0].set_title('Predicted')

sns.heatmap(df_actual_pivot, cmap='viridis', ax=axes[1],
            xticklabels=True, yticklabels=True)
axes[1].set_title('Actual')

plt.tight_layout()
plt.show()

In [ ]:
import wom

wom.predict_consumption_novel(taxonomy={"genus": "Pseudomonas", "family": "Pseudomonadaceae"})



In [ ]:
import wom
match = wom.resolve_taxon("variovorax")
print(match)
match = wom.resolve_metabolite("sucrose")
print(match)

In [ ]:
genera = ['Variovorax','Cupriavidus','Microbacterium','Brevundimonas','Pedobacter']
data = []
for genus in genera:
    temp = wom.consumed_by(genus=genus)
    temp['consumer_genus'] = genus
    data.append(temp)
import pandas as pd
df = pd.concat(data)
df = pd.pivot_table(df, index='chebi_name', columns='consumer_genus', values='mean_score', fill_value=0)
idx = df.max(axis=1)>5
df = df.loc[idx]
import seaborn as sns
import matplotlib.pyplot as plt
sns.clustermap(df, cmap='viridis')
plt.show()

In [ ]:
out = wom.secretes(chebi_id=15354)
out

In [ ]:
out = wom.consumes(chebi_id=17992)
out

In [ ]:
import wom

# Taxonomy-level queries
out = wom.consumed_by(genus="Serratia", top_k=20)
print(out.head(3))
out_secretes = wom.secreted_by(family="Rhodanobacteraceae", top_k=15)
print(out_secretes.head(3))

# Organism-level queries  
out = wom.predict_consumption("CPT56D-587-MTF", top_k=10)
print(out)
out = wom.similar_organisms("CPT56D-587-MTF")
print(out)
# Metabolite-level queries
out = wom.consumes(chebi_id=18050)  # glucose
print(out)
out = wom.secretes(chebi_id=28358)
print(out)
wom.cross_feeding_potential("CPT56D-587-MTF", "FW104-10B1")

# KG integration
# wom.paths_between("CHEBI:15903", "NCBITaxon:123456")

# Community queries
# wom.community_profile(["isolate1", "isolate2", "isolate3"])
# wom.niche_overlap("isolateA", "isolateB")


In [ ]:
import pandas as pd
f = '/pscratch/sd/b/bpb/wom_data_cache/cache/exometabolite.parquet'
df = pd.read_parquet(f)
df.head()

In [ ]:
import pandas as pd
kgm_nodes_filename = '/pscratch/sd/b/bpb/wom_data_cache/raw/kg-microbe-core/merged-kg/merged-kg_nodes.tsv'
kgm_edges_filename = '/pscratch/sd/b/bpb/wom_data_cache/raw/kg-microbe-core/merged-kg/merged-kg_edges.tsv'
mg_nodes_filename = '/pscratch/sd/b/bpb/wom_data_cache/raw/MetagenomicKG/MetagenomicKG/KG_nodes.tsv'
mg_edges_filename = '/pscratch/sd/b/bpb/wom_data_cache/raw/MetagenomicKG/MetagenomicKG/KG_edges.tsv'

my_files = [kgm_edges_filename, mg_edges_filename, kgm_nodes_filename, mg_nodes_filename]
for f in my_files:
    source = 'kg-microbe' if 'kg-microbe' in f else 'metagenomic-kg'
    data_type = 'edges' if 'edges' in f else 'nodes'
    print(f"Source: {source} | Data type: {data_type}")
    print(f"File: {f}")
    df = pd.read_csv(f, sep='\t', dtype=str,nrows=500000)
    if 'category' in df.columns:
        df.rename(columns={'category':'node_type'}, inplace=True)
    print(f"Columns: {df.columns}")
    print(df.sample(20).to_string(index=False))
    if data_type == 'nodes':
        print(f"Unique node categories: {df['node_type'].nunique()}")
        print(f"Node categories: {df['node_type'].value_counts().to_string()}")
    print("\n")